# iteration_patterns
Iteration patterns are repeatable shapes for walking data: scan, transform, filter, aggregate, pair, chunk, and validate. Strong engineers recognize these patterns quickly because they reduce reinvention and make loop behavior easier to reason about.

## 1. Problem First
Why does this matter in real systems?
- Data pipelines repeatedly apply the same iteration shapes to different domains.
- A messy loop often means the underlying iteration pattern was never made explicit.
- Choosing the wrong pattern can hurt both correctness and performance.

In [1]:
records = [{"latency": 120}, {"latency": 95}, {"latency": 140}]
slow_records = []
for record in records:
    if record["latency"] > 100:
        slow_records.append(record)
print(slow_records)

[{'latency': 120}, {'latency': 140}]


## 2. Minimal Concept
Common iteration patterns:
- Scan until found
- Filter selected records
- Transform each record
- Aggregate into counts or summaries
- Chunk into batches

## 3. Mental Model
How Python actually behaves
Most iteration code is a combination of a source iterable, a control condition, and an output shape. If you know whether you are searching, filtering, transforming, or aggregating, the loop becomes much easier to design and verify.

In [2]:
values = [10, 20, 30]
transformed = []
total = 0
for value in values:
    transformed.append(value * 2)
    total += value
print(transformed)
print(total)

[20, 40, 60]
60


## 4. Internal Mechanics
The mechanics are ordinary loops plus state: accumulators, flags, counters, or builders. What matters is that each pattern has a predictable state shape, which makes the code easier to test and refactor.

In [3]:
import dis

def count_errors(levels):
    total = 0
    for level in levels:
        if level == "ERROR":
            total += 1
    return total

dis.dis(count_errors)
print(count_errors(["INFO", "ERROR", "ERROR"]))

  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (0)
              4 STORE_FAST               1 (total)

  5           6 LOAD_FAST                0 (levels)
              8 GET_ITER
        >>   10 FOR_ITER                13 (to 40)
             14 STORE_FAST               2 (level)

  6          16 LOAD_FAST                2 (level)
             18 LOAD_CONST               2 ('ERROR')
             20 COMPARE_OP              40 (==)
             24 POP_JUMP_IF_TRUE         1 (to 28)
             26 JUMP_BACKWARD            9 (to 10)

  7     >>   28 LOAD_FAST                1 (total)
             30 LOAD_CONST               3 (1)
             32 BINARY_OP               13 (+=)
             36 STORE_FAST               1 (total)
             38 JUMP_BACKWARD           15 (to 10)

  5     >>   40 END_FOR

  8          42 LOAD_FAST                1 (total)
             44 RETURN_VALUE
2


## 5. Edge Cases
Where it breaks:
- Mixing too many patterns in one loop hides the real job.
- A search pattern with no early exit wastes work.
- A transform pattern that mutates inputs can leak side effects.
- Aggregation can silently use wrong initial values.

In [4]:
items = [1, 2, 3]
running_product = 0
for item in items:
    running_product *= item
print(running_product)

0


## 6. Performance Thinking
- Search-until-found can stop early and beat a full scan.
- Filter and transform patterns are usually O(n).
- Chunking can control memory use and external API pressure.
- Pattern clarity often reveals where a loop can be optimized or split.

## 7. Real Use Case
A metrics pipeline often filters invalid records, transforms fields, and aggregates totals in separate iteration steps.

In [5]:
events = [
    {"level": "INFO", "latency": 90},
    {"level": "ERROR", "latency": 120},
    {"level": "ERROR", "latency": 150}
]
error_latencies = []
for event in events:
    if event["level"] == "ERROR":
        error_latencies.append(event["latency"])
print(error_latencies)

[120, 150]


## 8. Anti-Pattern
What beginners do wrong:
- Write one giant loop that searches, mutates, logs, aggregates, and retries all at once.
- Recompute expensive values inside the loop unnecessarily.
- Pick a pattern accidentally instead of intentionally.

In [6]:
records = ["1", "bad", "3"]
total = 0
for record in records:
    if record.isdigit():
        total += int(record)
print(total)

4


## 9. Interview Signals
Questions FAANG asks:
- What iteration pattern best fits this problem?
- When should you split one loop into multiple clearer passes?
- How do you detect wasted work in a loop design?
- How do pattern choices affect complexity?

## 10. Exercise (Non-trivial)
Take a batch-processing loop that validates, filters, transforms, aggregates, and retries in one place. Break it into explicit iteration patterns, decide what should remain in one pass versus multiple passes, and justify the tradeoff between readability and performance.

In [7]:
def process_records(records):
    result = []
    total = 0
    for record in records:
        if record.get("valid"):
            record["latency"] *= 2
            total += record["latency"]
            result.append(record)
    return result, total

# Task:
# 1. Identify the iteration patterns hidden here.
# 2. Decide whether mutation is acceptable.
# 3. Refactor for clearer intent.